In [1]:
import matplotlib
matplotlib.use("Agg")
from scipy.integrate import odeint
from sympy import var, I
import numpy as np
import matplotlib.pyplot as plt
import math

import importlib
import sys
import os

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import set_style, colores, format_ax
set_style()

2026-03-18 17:19:07,086 - openket - INFO - openket v0.1.0 initialized successfully.


In [3]:
filename = "cavidad"

hbar = 1.0
rabi_b = 1.0 # tasa del bombeo, proporcional a la amplitud del campo eléctrico del láser
detuning = 0.0 # detuning entre frecuencia de la cavidad y frecuencia del bombeo

kappa = 1.0
nmax = 15 # truncación del espacio de Hilbert (número de estados de Fock)
dt = 1000
t = np.linspace(0, 15, dt)

base = [Ket(i, "cavidad") for i in range(nmax)]
rho = Operator("R")
a = AnnihilationOperator("cavidad", nmax-1)
aa = CreationOperator("cavidad", nmax-1)

H_c = hbar * detuning * (aa*a + 1/2) # Hamiltoniano del oscilador armónico cuántico (cavidad)
H_b = I*hbar * rabi_b * (aa - a) # Hamiltoniano del bombeo (láser)
H = H_c + H_b # Hamiltoniano total
# ecuación de movimiento de Lindblad
rdot = -I/hbar * comm(H,rho) + (kappa/2)*(2*a*rho*aa - aa*a*rho - rho*aa*a)

build_ode(
    rho=rho,
    rdot=rdot,
    basis=base,
    filetype="Scipy",
    filename=f"func{filename}.py",
    dictname=f"dic{filename}.py"
)

# leer módulo dic.py
funcname=f"func{filename}"
if funcname in sys.modules:
    del sys.modules[funcname]
modulo = importlib.import_module(funcname)
modulo = importlib.reload(modulo)
f = modulo.f
dic = modulo.dic
# convertir condiciones iniciales simbólicas -> numéricas
rho0 = Ket(0,"cavidad") * dag(Ket(0,"cavidad"))
init_conditions = init_state(rho=rho, rho0=rho0, basis=base, dic=dic)
# solución numérica
rho_solution = odeint(f, init_conditions, t)

In [4]:
# Valores esperados numéricos
# definición simbólica de los observables
N = aa * a # operador de número
N_symb = sub_qexpr(qexpr=trace(rho * N, basis=base), dic=dic)
N_expect = sym2num(sol=rho_solution, symbexpr=N_symb)
N_expect = np.real(N_expect)

In [5]:
# Valores esperados analíticos
alpha_ss = rabi_b / (kappa/2 + 1j*detuning) #amplitud compleja del estado estacionario (cuando t->oo)
alpha = alpha_ss*(1 - np.exp(-(kappa/2 + 1j*detuning) * t)) #amplitud compleja del campo coherente en la cavidad

N_analytical = np.abs(alpha)**2

In [11]:
step=20
plt.scatter(t[::step], N_expect[::step], label=r'numérico', color=colores[3], s=15, marker='s', zorder=2)
plt.plot(t, N_analytical, label=r'analítico', color=colores[4], linewidth=1, zorder=1)
plt.xlabel(r"Tiempo adimensional $\Omega_p t$")
plt.ylabel(r"Número medio de fotones")
plt.axhline(4, color="gray", linestyle="--", linewidth=0.5)
ax = plt.gca()
format_ax(ax, xstep=2, ystep=2, ymax=max(N_expect)+1)
plt.legend()
plt.savefig(f"../figuras/fig:conv_cavidad_disipacion.png")
plt.close()

for file in os.listdir("."):
    if file.endswith(".py") and file != "style.py":
        os.remove(file)